# Framework demonstration for runtime enforcement

1. A comprehensive enforcement intercepts the entire agent lifecycle across several key stages:

    a) **Tool Execution** - before/after tool calls
    b) **Planning** - during reasoning and decision-making phases (e.g., Chain-of thought reasoning)
    c) **Memory Operations** - during retrieval and write operations (e.g., Retrieval Augmented Generation)
    d) **User Interaction** - at input reception and output generation.

This holistic interception provides control points at all critical components of agent behavior, enabling enforcement of policies, guardrails, security constraints, and compliance requirements across the full agent architecture.

In our framework, such event is specified by trigger:

In [ ]:
# rules for simply stop at specific event (e.g.,)

rule_before_tool_execution = """
rule@stop_before_tool
trigger
    before_action
check
    True
enforce
    stop
"""

Using the above rule, we can control the agent stop before tool execution.

We first set up an agent that can access python interpreter.

In [5]:
!pip install -r ../requirement.txt

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/ed/5c/5c0be747261e1f8129b875fa3bfea736bc5fe17652f9d5e15ca118571b6f/langchain-0.3.25-py3-none-any.whl (1.0 MB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/d8/d5/c90c5478215c20ee71d8feaf676f7ffd78d0568f8c98bd83f81ce7562ed7/langchain_openai-0.3.35-py3-none-any.whl (75 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/e1/e6/c249b20573ec98c427956327387ef23d76d4399c39ae08291add85601b1b/virustotal_python-1.1.0-py3-none-any.whl (8.0 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/b8/29/60802a71c4a0b573c0e9ecda1846404b04bdbc9bc805732b39261be4c376/langchain_core-0.3.81-py3-none-any.whl (457 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/58/0d/41a51b40d24ff0384ec4f7ab8dd3dcea8353c05c973836b5e289f1465d4f/langchain_text_splitters-0.3.11-py3-none-any.whl (33 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/6a/f4/c20

In [9]:
!pip list | grep langchain

langchain                0.3.25
langchain-classic        1.0.1
langchain-community      0.3.25
langchain-core           0.3.81
langchain-experimental   0.3.4
langchain-openai         0.3.35
langchain-text-splitters 0.3.11


In [1]:
import langchain
print(langchain.__version__)


0.3.25


In [ ]:

from controlled_agent_excector import initialize_controlled_agent 
from langchain_experimental.utilities import PythonREPL
from langchain_openai import ChatOpenAI

from langchain_core.agents import AgentAction, AgentFinish, AgentStep
from langchain.agents import initialize_agent, types
# from langchain.agents.agent_types import AgentType
from langchain.tools import tool, Tool

with open("../key.txt") as f:
    key = f.read()

# Initialize the LLM
llm = ChatOpenAI(model = "gpt-4o", api_key=key)

repl_tool = Tool(
    name="python_repl",
    description="A Python shell. Use this to execute python commands. Input should be a valid python command. If you want to see the output of a value, you should print it out with `print(...)`.",
    func=PythonREPL().run
)

tools = [repl_tool]

An agent use case without runtime enforcement:

In [ ]:
agent = initialize_controlled_agent(tools, llm, agent="zero-shot-react-description", rules = [])

res = agent.invoke("what is 1.123+1.432?")
print(res)

Python REPL can execute arbitrary code. Use with caution.


{'input': 'what is 1.123+1.432?', 'output': '1.123 + 1.432 = 2.555', 'intermediate_steps': [(AgentAction(tool='python_repl', tool_input='print(1.123 + 1.432)', log='To find the sum of 1.123 and 1.432, I will perform a simple arithmetic operation by adding these two numbers together.\n\nAction: python_repl\nAction Input: print(1.123 + 1.432)'), '2.5549999999999997\n')]}


A use case that ask for user confirmation before executing a tool:



2. Demonstrating runtime enforcement framework.
 2.1 assume we have some predefined safety rules, how can we enforce the rule.
 2.2 assume we have some high level safety goals, how can we obtain rules from the samples. (new research question: how do we validate and refine the rule)

3. Demonstrating proactive runtime enforcement framework
 3.1 assume we have some predefined sensitive states, simulate to collect traces to model agent behaviour
 3.2 using the model, we can predict the future risk and take enforcements.